<a href="https://colab.research.google.com/github/mvashi-sonic/AICapstoneProj/blob/dev/capstone_FAISS_retriever.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install sentence-transformers faiss-cpu

Mount The Drive

In [2]:
from google.colab import drive

# 1. Mount your Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


Import Required Packages

In [3]:
import json
import pickle
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

Sentence Tranformer

In [ ]:
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)
print(embedder.get_embedding_dimension())

Read the Data

In [5]:
chunks = []
dict = {}
#with open("/content/drive/MyDrive/AI_capstone_training_data/all_records_final_formatted_3.json", encoding="utf-8") as f:
test_rec = open("test_records.jsonl", "a")
with open("/content/drive/MyDrive/AI_capstone_training_data/positive_records_singular.json", encoding="utf-8") as f:
    records = json.load(f)
    #for record in records["All_Records"]:
    for record in records["Positive_Records"]:
      if(record["chunk_id"] not in dict):
        dict[record["chunk_id"]] = 1
        chunks.append(record)

    print(f"Distinct Records: {len(chunks)}")
json.dump(chunks,test_rec )

Distinct Records: 20691


Create Pragraphs With Metadata

In [ ]:
paragraphs = [
    chunk["reference"]
    for chunk in chunks
]
print(paragraphs[0][:300])
print(paragraphs[1][:300])
print(paragraphs[2][:300])

Create Embeddings

In [7]:


embeddings = embedder.encode(
    paragraphs,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

embeddings = np.asarray(
    embeddings,
    dtype=np.float32
)
print(embeddings.shape)

Batches:   0%|          | 0/162 [00:00<?, ?it/s]

(20691, 384)


In [ ]:

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)
print("FAISS vectors:", index.ntotal)
print("Number of chunks:", len(chunks))
print("Number of embeddings:", len(embeddings))


Write Indexes

In [9]:


faiss.write_index(
    index,
    "financial_reports.index"
)

In [10]:
#Check the file

import os

size_mb = os.path.getsize(
    "financial_reports.index"
) / (1024 * 1024)

print(f"Index size: {size_mb:.2f} MB")

Index size: 30.31 MB


In [11]:
metadata = []

for chunk in chunks:
    #print(f"{chunk["metadata"]}")

    metadata.append({
        "chunk_id": chunk["chunk_id"],
        "reference": chunk["reference"]
    })

In [ ]:
with open("financial_reports_metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

In [13]:
import os

print(
    "File size:",
    os.path.getsize("financial_reports.index"),
    "bytes"
)

File size: 31781421 bytes


Read Indexes

In [7]:
index = faiss.read_index(
    "/content/drive/MyDrive/capstone_FAISS_embeddings/financial_reports.index"
)
with open("/content/drive/MyDrive/capstone_FAISS_embeddings/financial_reports_metadata.pkl","rb") as f:
    metadata = pickle.load(f)


Search For Top 50 Answers Based On Question Encoding

In [8]:
question = "What is Apple's revenue"

query_embedding = embedder.encode(
    [question],
    convert_to_numpy=True,
    normalize_embeddings=True
)

query_embedding = np.asarray(
    query_embedding,
    dtype=np.float32
)

scores, indices = index.search(
    query_embedding,
    50
)

print("Unique indices:", len(set(indices[0])))
print(indices[0])

Unique indices: 50
[ 350  351   32    6  359   55   54  360   53   12   10  353  368  358
  378   39  369  337    5   56  103  357   35  110  324   34  347  354
   60   65   61   51   46 3028   41  219   83  367  340   74   73  355
  362  361   84   81  365  371  352   43]


In [9]:
scores, ids = index.search(
    query_embedding,
    k=50
)
print(scores)
print(ids)

[[0.6725097  0.6603534  0.6573678  0.64513385 0.64182204 0.61445886
  0.6055928  0.6052366  0.6005393  0.5980278  0.5901251  0.58750176
  0.5833433  0.57899183 0.57669085 0.5723034  0.57048744 0.5675676
  0.56656104 0.5645948  0.56188196 0.55925786 0.558082   0.55777586
  0.554137   0.5535671  0.5534787  0.5523194  0.5521052  0.5517931
  0.55169666 0.55132926 0.54915917 0.54818785 0.5474608  0.54576504
  0.54546756 0.5452055  0.54331344 0.54305387 0.53878534 0.53866494
  0.5386066  0.53564113 0.5345828  0.5341631  0.5322534  0.53074896
  0.5291139  0.52872294]]
[[ 350  351   32    6  359   55   54  360   53   12   10  353  368  358
   378   39  369  337    5   56  103  357   35  110  324   34  347  354
    60   65   61   51   46 3028   41  219   83  367  340   74   73  355
   362  361   84   81  365  371  352   43]]


In [ ]:
for score, idx in zip(scores[0], ids[0]):
    chunk = metadata[idx]

    file_output = open("retrieved_results1.jsonl", "a")
    dict = {}
    dict["score"] = f"{score:.4f}"
    """dict["ticker"] = chunk["ticker"]
    dict["section"] = chunk["section"]
    dict["year"] = chunk["section"]"""
    dict["reference"] = chunk["reference"]
    dict["chunk_id"] = chunk["chunk_id"]
    print(dict["reference"])
    print(dict["chunk_id"])
    json.dump(dict, file_output)





In [ ]:
for rank, (score, ids) in enumerate(zip(scores[0], indices[0])):
    print(
        f"{rank+1}: idx={ids}, score={score:.4f}"
    )

Copy Results To Drive

In [16]:
#keep
#Copy data to drive
import os
import shutil
from google.colab import drive

# 2. Define source and destination folders
# Replace 'my_folder' with the exact folder path where your .json files currently are
source_dir = '.'
# Replace 'My Drive/TargetFolder' with the Drive folder you want to copy to
destination_dir = '/content/drive/MyDrive/AI_capstone_training_data/FAISS_Retriever'

# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# 3. Find and copy all .json files
for filename in os.listdir(source_dir):
    if filename.endswith('retrieved_results.json'):
        source_file = os.path.join(source_dir, filename)
        destination_file = os.path.join(destination_dir, filename)

        shutil.copy2(source_file, destination_file)
        print(f"Copied: {filename}")

print("All .json files copied successfully!")

All .json files copied successfully!


Evaluate FAISS

In [ ]:
import numpy as np
eval_records = chunks

"""test_file = open("test_records.jsonl", "a")
for line in test_file:
         j_obj = json.loads(line)
         eval_records.append[j_obj]"""

def evaluate_faiss(
    eval_records,
    embedder,
    index,
    metadata,
    #k_values=(1, 5, 10, 20)
    k_values=(80, 100)
):
    hits = {k: 0 for k in k_values}
    reciprocal_ranks = []

    max_k = max(k_values)

    for record in eval_records:
        question = record["question"]
        correct_chunk_id = record["chunk_id"]

        # Encode query
        query_embedding = embedder.encode(
            [question],
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype(np.float32)

        # Retrieve candidates
        scores, ids = index.search(
            query_embedding,
            max_k
        )

        retrieved_ids = [
            metadata[idx]["chunk_id"]
            for idx in ids[0]
            if idx != -1
        ]

        # Hit / Recall @ K
        for k in k_values:
            if correct_chunk_id in retrieved_ids[:k]:
                hits[k] += 1

        # Reciprocal rank
        if correct_chunk_id in retrieved_ids:
            rank = retrieved_ids.index(correct_chunk_id) + 1
            reciprocal_ranks.append(1.0 / rank)
        else:
            reciprocal_ranks.append(0.0)

    n = len(eval_records)

    metrics = {}

    for k in k_values:
        metrics[f"Recall@{k}"] = hits[k] / n

    metrics["MRR"] = sum(reciprocal_ranks) / n
    print(metrics)
    return metrics

evaluate_faiss(eval_records,
    embedder,
    index,
    metadata,

)